## rknn_toolkit2 适配 python3.10, 所以在运行前请更换 kernel

In [1]:
%pip install torch==2.4.0 torchvision==0.19.0 -i https://pypi.tuna.tsinghua.edu.cn/simple
%pip install setuptools==80.9.0 -i https://pypi.tuna.tsinghua.edu.cn/simple
%pip install onnx==1.18 onnxruntime==1.18 -i https://pypi.tuna.tsinghua.edu.cn/simple
%pip install rknn_toolkit2 -i https://pypi.tuna.tsinghua.edu.cn/simple

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 641.2 kB/s  0:00:270:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 701.4 kB/s  0:00:09 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [onnxruntime] [onnxruntime]
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Note: you may need to restart the kernel to use updated packages.


### 数据集 MNIST 处理

In [8]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

test_dataset = datasets.MNIST(
    root="./dataset",
    train=False,
    download=True,
    transform=transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1000,
    shuffle=False
)

print("测试集大小:", len(test_dataset))
print("batch 数量:", len(test_loader))

测试集大小: 10000
batch 数量: 10


### ONNX 准确率

In [9]:
import numpy as np
import onnxruntime as ort

def onnxAccuracy(model):
    session = ort.InferenceSession(
        model,
        providers=["CPUExecutionProvider"]
    )

    input_name = session.get_inputs()[0].name
    output_name = session.get_outputs()[0].name

    correct = 0
    total = 0

    for images, labels in test_loader:
        images = images.numpy().astype(np.float32)
        outputs = session.run(
            [output_name],
            {input_name: images}
        )[0]
        predictions = np.argmax(outputs, axis=1)
        labels = labels.numpy()
        correct += np.sum(predictions == labels)
        total += labels.shape[0]

    accuracy = correct / total * 100

    print(f"ONNX Accuracy: {accuracy:.2f}%")

onnxAccuracy("./model/model_pruned_fp32.onnx")

ONNX Accuracy: 97.32%


### 校准数据集

In [ ]:
import os
from torchvision import datasets

DATASET_DIR = "./dataset"
CALIB_DIR = "./calibration"
DATASET_TXT = "./calibration.txt"

os.makedirs(CALIB_DIR, exist_ok=True)

dataset = datasets.MNIST(
    root=DATASET_DIR,
    train=True,
    download=False
)

num_calib = 200

with open(DATASET_TXT, "w") as f:
    for i in range(num_calib):
        img, label = dataset[i]

        path = os.path.join(
            CALIB_DIR,
            f"{i:05d}.png"
        )

        img.save(path)

        # dataset.txt 每行一张图片
        f.write(path + "\n")

print("校准数据生成完成")
print("图片数量:", num_calib)

### INT8 PTQ + 计算准确率

In [29]:
from rknn.api import RKNN

ONNX_PATH = "./model/model_pruned_fp32.onnx"
RKNN_PATH = "./model/model_pruned_int8.rknn"
CALIBRATION_PATH = "./calibration.txt"

rknn = RKNN(verbose=True)

ret = rknn.config(
    target_platform="rk3588",
    mean_values=[[0.1307*255]],
    std_values=[[0.3081*255]],
)
if ret != 0:
    print("config failed:", ret)
    exit(ret)

ret = rknn.load_onnx(
    model=ONNX_PATH,
    inputs=["input"],
    input_size_list=[[1, 1, 28, 28]],
)
if ret != 0:
    print("load_onnx failed:", ret)
    exit(ret)

ret = rknn.build(
    do_quantization=True,
    dataset=CALIBRATION_PATH,
)
if ret != 0:
    print("build failed:", ret)
    exit(ret)

ret = rknn.export_rknn(
    RKNN_PATH,
)
if ret != 0:
    print("export_rknn failed:", ret)
    exit(ret)

print("\n########################")
print("\nINT8 PTQ successfully")
print("\n########################")

##################
# RKNN 准确率
##################

ret = rknn.init_runtime()
if ret != 0:
    print("init_runtime failed:", ret)
    exit(ret)

correct = 0
total = len(test_dataset)

transform = transforms.Compose([
    transforms.ToTensor(),
])

test_dataset = datasets.MNIST(
    root="./dataset",
    train=False,
    download=True,
    transform=transform
)

for i in range(total):
    image, label = test_dataset[i]
    image = image.numpy().astype(np.float32)
    image = np.expand_dims(
        image,
        axis=0
    )
    outputs = rknn.inference(
        inputs=[image],
        data_format=["nchw"],
    )
    
    if outputs is None:
        print(f"Inference failed at sample {i}")
        
    output = np.asarray(outputs[0]).reshape(-1)
    pred = np.argmax(output)

    if pred == label:
        correct += 1

accuracy = correct / total * 100

print("\n########################")
print(f"\nRKNN Accuracy: {accuracy:.2f}%")
print("\n########################")

rknn.release()

I rknn-toolkit2 version: 2.3.2
W load_onnx: If you don't need to crop the model, don't set 'inputs'/'input_size_list'/'outputs'!
I Loading : 100%|██████████████████████████████████████████████████| 9/9 [00:00<00:00, 11414.80it/s]
D base_optimize ...
D base_optimize done.
D 
D fold_constant ...
D fold_constant done.
D fold_constant remove nodes = ['node_Concat_4', 'node_Shape_0']
D Fixed the shape information of some tensor!
D 
D correct_ops ...
D correct_ops done.
D 
D fuse_ops ...
D fuse_ops results:
D     convert_gemm_by_exmatmul: remove node = ['node_linear'], add node = ['view_tp', 'view_tp_rs', 'node_linear#1', 'linear_mm_tp', 'linear_mm_tp_rs']
D     unsqueeze_to_4d_relu: remove node = [], add node = ['linear_rs', 'relu_2-rs']
D     convert_gemm_by_exmatmul: remove node = ['node_linear_1'], add node = ['relu_2_tp', 'relu_2_tp_rs', 'node_linear_1#1', 'linear_1_mm_tp', 'linear_1_mm_tp_rs']
D     unsqueeze_to_4d_logsoftmax: remove node = [], add node = ['linear_1_rs', 'output-rs']
D

200
D RKNN: [19:05:43.725] 6   Conv     fc2.weight    INT8      (10,38,1,1)   | 0x0001ab40 0x0001ad20 0x000001e0
D RKNN: [19:05:43.725] 6   Conv     fc2.bias      INT32     (10)          | 0x0001ad40 0x0001ae40 0x00000100
D RKNN: [19:05:43.725] 8   Reshape  output-rs_i1  INT64     (2)           | 0x0001ae80 0x0001ae90 0x00000010
D RKNN: [19:05:43.725] ---------------------------------------------------+---------------------------------
D RKNN: [19:05:43.725] ----------------------------------------
D RKNN: [19:05:43.725] Total Internal Memory Size: 40KB
D RKNN: [19:05:43.725] Total Weight Memory Size: 109.688KB
D RKNN: [19:05:43.725] ----------------------------------------
D RKNN: [19:05:43.725] <<<<<<<< end: rknn::RKNNMemStatisticsPass
I RKNN: [19:08:35.171] compress = 0, conv_eltwise_activation_fuse = 1, global_fuse = 1, multi-core-model-mode = 7, output_optimize = 1, layout_match = 1, enable_argb_group = 0, op_group_sram_opt = 0, enable_flash_attention = 0, op_group_nbuf_opt = 0, s

I SessionPreparing : 100%|████████████████████████████████████████| 13/13 [00:00<00:00, 2224.91it/s]



########################

RKNN Accuracy: 11.35%

########################


### 计算内存大小

In [27]:
import os

path = "./model/model_pruned_int8.rknn"

size_bytes = os.path.getsize(path)
size_mb = size_bytes / 1024 / 1024

print(f"RKNN模型大小: {size_mb:.2f} MB")

RKNN模型大小: 0.14 MB
